In [5]:
import random
from typing import Iterable
import numpy as np
import pandas as pd
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import train_test_split
from datasets import load_dataset

SEED = 42

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


Device: cpu


In [8]:
MODEL_NAME = "nateraw/bert-base-uncased-emotion"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME).to(device)
model.eval()

print("Model loaded:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)
print("Model class:", model.__class__.__name__)
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 39693.73it/s]
BertForSequenceClassification LOAD REPORT from: nateraw/bert-base-uncased-emotion
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded: nateraw/bert-base-uncased-emotion
Tokenizer class: BertTokenizer
Model class: BertForSequenceClassification
Number of labels: 6
id2label: {0: 'sadness', 1: 'joy', 2: 'love', 3: 'anger', 4: 'fear', 5: 'surprise'}


In [7]:

def load_emotion_dataset(dataset_name="dair-ai/emotion", split_ratios=(0.7, 0.15, 0.15), seed=SEED):
    """
    Загружает датасет для классификации эмоций и разбивает на train/val/test.
    """
    print(f"Загрузка датасета: {dataset_name}")
    
    dataset = load_dataset(dataset_name)
    
    full_df = None
    
    if 'train' in dataset:
        full_df = dataset['train'].to_pandas()
    elif 'validation' in dataset:
        # Иногда датасет сразу разбит на train/validation/test
        # В таком случае объединяем их или берем validation как основной
        full_df = dataset['validation'].to_pandas()
    elif 'default' in dataset:
        full_df = dataset['default'].to_pandas()
    else:
        # Берем первый доступный сплит
        first_key = list(dataset.keys())[0]
        full_df = dataset[first_key].to_pandas()
    
    # Проверка на случай, если датасет пуст
    if full_df is None or len(full_df) == 0:
        raise ValueError("Не удалось загрузить данные из датасета")

    # 2. Разбиваем на сплиты
    train_ratio, val_ratio, test_ratio = split_ratios
    
    # Сначала разбиваем на train+val и test
    train_val_df, test_df = train_test_split(
        full_df, 
        test_size=test_ratio, 
        random_state=seed,
        stratify=full_df['label'] if 'label' in full_df.columns else None
    )
    
    # Затем разбиваем train+val на train и val
    # val_ratio от оставшейся части (train_val)
    adjusted_val_ratio = val_ratio / (train_ratio + val_ratio)
    
    train_df, val_df = train_test_split(
        train_val_df, 
        test_size=adjusted_val_ratio, 
        random_state=seed,
        stratify=train_val_df['label'] if 'label' in train_val_df.columns else None
    )

    # Sanity-check:
    print(f"Размер полного датаcета: {len(full_df)}")
    
    print(f"\nРазмеры датасетов:")
    print(f"  Train: {len(train_df)} samples ({len(train_df)/len(full_df)*100:.1f}%)")
    print(f"  Val:   {len(val_df)} samples ({len(val_df)/len(full_df)*100:.1f}%)")
    print(f"  Test:  {len(test_df)} samples ({len(test_df)/len(full_df)*100:.1f}%)")
    
    if 'label' in train_df.columns:
        print(f"\nРаспределение классов в train:")
        print(train_df['label'].value_counts().sort_index())

    emotion_labels = {
        0: 'sadness',
        1: 'neutral', 
        2: 'joy',
        3: 'love',
        4: 'anger',
        5: 'fear',
        6: 'surprise'
    }

    print(f"Метки классов: {emotion_labels}")
    
    return train_df, val_df, test_df


train_df, val_df, test_df = load_emotion_dataset(
    split_ratios=(0.7, 0.15, 0.15),
)

# Показываем примеры данных
print("Примеры данных из train:")
print(train_df.head())

Загрузка датасета: dair-ai/emotion
Размер полного датаcета: 16000

Размеры датасетов:
  Train: 11200 samples (70.0%)
  Val:   2400 samples (15.0%)
  Test:  2400 samples (15.0%)

Распределение классов в train:
label
0    3266
1    3754
2     913
3    1511
4    1356
5     400
Name: count, dtype: int64
Метки классов: {0: 'sadness', 1: 'neutral', 2: 'joy', 3: 'love', 4: 'anger', 5: 'fear', 6: 'surprise'}
Примеры данных из train:
                                                    text  label
10703    i am feeling more like me except a little weepy      0
6396   i feel disturbed because of the world i saw th...      0
10887  i feel i am suffering from a bad case of i onl...      0
1641   i feel like the audience is smart enough and k...      1
7281   i wake up it hurts knowing that i could have e...      1


Датасет содержит короткие отрывки текста. Задача заключаетс в том, чтобы классифицировать их по эмоциональной окраске.